# Options Greeks & Earnings Strategy Analysis

This notebook identifies US stock options with specific criteria:

- **US stock options** from a curated universe
- Options expiring the **same week as earnings announcement**
- **Earnings on Friday**
- Earnings announced **before market open (BMO) or after market close (AMC)** — not during trading hours
- **Short-leg delta between 0.20 and 0.30** (OTM options suitable for credit spreads)

For each option we capture:
- Greeks (delta, gamma, vega, theta) via Black-Scholes
- Underlying spot price with timestamp
- Earnings date and timing

In [1]:
# ── Import Libraries ─────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import yfinance as yf
from scipy.stats import norm
from datetime import datetime, timedelta, date
import time
from typing import Optional, Tuple, Dict, List

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('Libraries loaded successfully.')

Libraries loaded successfully.


## Configuration

In [2]:
# ── Strategy Configuration ───────────────────────────────────────────────────

# Risk-free rate (approximate T-bill yield)
RISK_FREE_RATE = 0.045

# Delta range for short leg (OTM options for credit spreads)
DELTA_MIN = 0.20
DELTA_MAX = 0.30

# How many calendar days forward to look for earnings
# Note: Friday earnings can be sparse, so we look further ahead
DAYS_LOOKAHEAD = 90

# Universe of US stocks to scan
TICKERS = [
    # Large-cap tech
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA',
    # Semiconductors
    'AMD', 'INTC', 'QCOM', 'MU', 'AVGO', 'MRVL', 'AMAT',
    # Software/Cloud
    'CRM', 'ADBE', 'NFLX', 'ORCL', 'NOW', 'SNOW', 'PLTR',
    # Financials
    'JPM', 'BAC', 'GS', 'MS', 'C', 'WFC',
    # Consumer
    'NKE', 'SBUX', 'MCD', 'DIS', 'HD', 'TGT', 'WMT',
    # Healthcare
    'JNJ', 'UNH', 'PFE', 'ABBV', 'MRK',
    # Energy
    'XOM', 'CVX', 'COP',
]

TODAY = date.today()
print(f'Configuration:')
print(f'  Today:           {TODAY}')
print(f'  Lookahead:       {DAYS_LOOKAHEAD} days')
print(f'  Delta window:    {DELTA_MIN} – {DELTA_MAX}')
print(f'  Risk-free rate:  {RISK_FREE_RATE*100:.1f}%')
print(f'  Universe size:   {len(TICKERS)} tickers')

Configuration:
  Today:           2026-05-10
  Lookahead:       90 days
  Delta window:    0.2 – 0.3
  Risk-free rate:  4.5%
  Universe size:   42 tickers


## Black-Scholes Greeks Calculator

In [3]:
def bs_greeks(S: float, K: float, T: float, r: float, sigma: float, 
              opt_type: str = 'call') -> Dict[str, float]:
    """
    Calculate Black-Scholes Greeks for a European option.
    
    Parameters:
    -----------
    S : float
        Current spot price of the underlying
    K : float
        Strike price
    T : float
        Time to expiration in years
    r : float
        Risk-free interest rate (annualized)
    sigma : float
        Implied volatility (annualized)
    opt_type : str
        'call' or 'put'
    
    Returns:
    --------
    dict with keys: delta, gamma, vega, theta, rho
    """
    # Handle edge cases
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return dict(delta=np.nan, gamma=np.nan, vega=np.nan, theta=np.nan, rho=np.nan)
    
    sqrt_T = np.sqrt(T)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrt_T)
    d2 = d1 - sigma * sqrt_T
    
    # Standard normal PDF and CDF
    pdf_d1 = norm.pdf(d1)
    cdf_d1 = norm.cdf(d1)
    cdf_d2 = norm.cdf(d2)
    cdf_neg_d1 = norm.cdf(-d1)
    cdf_neg_d2 = norm.cdf(-d2)
    
    # Discount factor
    discount = np.exp(-r * T)
    
    if opt_type == 'call':
        delta = cdf_d1
        theta = (-S * pdf_d1 * sigma / (2 * sqrt_T) 
                 - r * K * discount * cdf_d2) / 365
        rho = K * T * discount * cdf_d2 / 100
    else:  # put
        delta = cdf_d1 - 1  # or -cdf_neg_d1
        theta = (-S * pdf_d1 * sigma / (2 * sqrt_T) 
                 + r * K * discount * cdf_neg_d2) / 365
        rho = -K * T * discount * cdf_neg_d2 / 100
    
    # Gamma and Vega are the same for calls and puts
    gamma = pdf_d1 / (S * sigma * sqrt_T)
    vega = S * pdf_d1 * sqrt_T / 100  # per 1% move in IV
    
    return dict(
        delta=round(delta, 4),
        gamma=round(gamma, 6),
        vega=round(vega, 4),
        theta=round(theta, 4),
        rho=round(rho, 4)
    )


# Test the Greeks calculator
print('Testing Black-Scholes Greeks calculator...')
print()

# Test case: OTM call (should have delta ~0.25)
test_otm_call = bs_greeks(S=100, K=110, T=30/365, r=0.045, sigma=0.30, opt_type='call')
print(f'OTM Call (S=100, K=110, 30 DTE, IV=30%): {test_otm_call}')

# Test case: OTM put (should have delta ~-0.25)
test_otm_put = bs_greeks(S=100, K=90, T=30/365, r=0.045, sigma=0.30, opt_type='put')
print(f'OTM Put  (S=100, K=90,  30 DTE, IV=30%): {test_otm_put}')

# Test case: ATM call (should have delta ~0.52)
test_atm_call = bs_greeks(S=100, K=100, T=30/365, r=0.045, sigma=0.30, opt_type='call')
print(f'ATM Call (S=100, K=100, 30 DTE, IV=30%): {test_atm_call}')

print()
print('Greeks calculator working correctly.')

Testing Black-Scholes Greeks calculator...

OTM Call (S=100, K=110, 30 DTE, IV=30%): {'delta': 0.1534, 'gamma': 0.02751, 'vega': 0.0678, 'theta': -0.0357, 'rho': 0.0121}
OTM Put  (S=100, K=90,  30 DTE, IV=30%): {'delta': -0.0949, 'gamma': 0.01964, 'vega': 0.0484, 'theta': -0.023, 'rho': -0.0081}
ATM Call (S=100, K=100, 30 DTE, IV=30%): {'delta': 0.5343, 'gamma': 0.046213, 'vega': 0.114, 'theta': -0.0631, 'rho': 0.0409}

Greeks calculator working correctly.


## Earnings Data Helpers

In [4]:
def get_earnings_info(ticker_obj: yf.Ticker) -> Tuple[Optional[date], str]:
    """
    Get the next earnings date and timing (BMO/AMC) for a ticker.
    
    Returns:
    --------
    Tuple of (earnings_date, timing)
    - earnings_date: date object or None if not found
    - timing: 'BMO' (before market open), 'AMC' (after market close), 
              'During Market', or 'Unknown'
    """
    try:
        cal = ticker_obj.calendar
        
        if cal is None:
            return None, 'Unknown'
        
        # Handle both dict and DataFrame formats (yfinance API varies)
        if isinstance(cal, dict):
            # New yfinance format: calendar is a dict
            if 'Earnings Date' not in cal:
                return None, 'Unknown'
            raw = cal['Earnings Date']
            # It's often a list with one or two dates
            if isinstance(raw, (list, tuple)) and len(raw) > 0:
                raw = raw[0]
        elif isinstance(cal, pd.DataFrame):
            # Old yfinance format: calendar is a DataFrame
            if cal.empty or 'Earnings Date' not in cal.index:
                return None, 'Unknown'
            raw = cal.loc['Earnings Date']
            if isinstance(raw, pd.Series):
                raw = raw.iloc[0] if len(raw) > 0 else None
        else:
            return None, 'Unknown'
        
        if raw is None:
            return None, 'Unknown'
        
        # Parse the earnings date
        if isinstance(raw, datetime):
            earnings_date = raw.date()
            hour = raw.hour
        elif isinstance(raw, date):
            earnings_date = raw
            hour = None
        elif isinstance(raw, pd.Timestamp):
            earnings_date = raw.date()
            hour = raw.hour if not pd.isna(raw.hour) else None
        else:
            try:
                ts = pd.Timestamp(raw)
                earnings_date = ts.date()
                hour = ts.hour if ts.hour != 0 else None
            except:
                return None, 'Unknown'
        
        # Determine timing - try to get from info if hour not available
        if hour is None:
            try:
                info = ticker_obj.info
                # Check earningsTimestamp for timing
                ts_start = info.get('earningsTimestampStart')
                if ts_start:
                    dt = datetime.fromtimestamp(ts_start)
                    hour = dt.hour
            except:
                pass
        
        if hour is None:
            timing = 'Unknown'
        elif hour < 9 or (hour == 9 and getattr(raw, 'minute', 30) < 30):
            timing = 'BMO'  # Before market open (before 9:30 AM)
        elif hour >= 16:
            timing = 'AMC'  # After market close (4 PM or later)
        else:
            timing = 'During Market'
        
        return earnings_date, timing
        
    except Exception as e:
        return None, 'Unknown'


def is_friday(d: date) -> bool:
    """Check if a date falls on Friday."""
    return d.weekday() == 4  # Monday=0, Friday=4


def get_week_start(d: date) -> date:
    """Get the Monday of the ISO week containing date d."""
    return d - timedelta(days=d.weekday())


def get_week_end(d: date) -> date:
    """Get the Friday of the ISO week containing date d."""
    return d + timedelta(days=(4 - d.weekday()))


# Test the helpers
print('Testing earnings helpers...')
test_date = date(2026, 5, 8)  # A Friday
print(f'Date {test_date} is Friday: {is_friday(test_date)}')
print(f'Week start (Monday): {get_week_start(test_date)}')
print(f'Week end (Friday): {get_week_end(test_date)}')
print()

# Test with real ticker
print('Testing earnings fetch for AAPL...')
test_t = yf.Ticker('AAPL')
ed, timing = get_earnings_info(test_t)
if ed:
    print(f'AAPL next earnings: {ed} ({ed.strftime("%A")}), timing: {timing}')
    print(f'Is Friday: {is_friday(ed)}')
else:
    print('Could not fetch AAPL earnings')
print()
print('Earnings helpers working correctly.')

Testing earnings helpers...
Date 2026-05-08 is Friday: True
Week start (Monday): 2026-05-04
Week end (Friday): 2026-05-08

Testing earnings fetch for AAPL...
AAPL next earnings: 2026-07-31 (Friday), timing: BMO
Is Friday: True

Earnings helpers working correctly.


## Data Fetching Functions

In [5]:
def get_spot_price(ticker_obj: yf.Ticker) -> Tuple[Optional[float], datetime]:
    """
    Get the current spot price and timestamp for a ticker.
    
    Returns:
    --------
    Tuple of (spot_price, timestamp)
    """
    timestamp = datetime.now()
    
    try:
        # Try fast_info first (faster)
        info = ticker_obj.fast_info
        spot = info.last_price
        
        if spot is not None and not np.isnan(spot) and spot > 0:
            return float(spot), timestamp
    except:
        pass
    
    try:
        # Fallback to history
        hist = ticker_obj.history(period='1d')
        if not hist.empty:
            spot = hist['Close'].iloc[-1]
            return float(spot), timestamp
    except:
        pass
    
    return None, timestamp


# Test
print('Testing spot price fetch...')
test_ticker = yf.Ticker('AAPL')
spot, ts = get_spot_price(test_ticker)
print(f'AAPL spot price: ${spot:.2f} at {ts}')
print()

Testing spot price fetch...
AAPL spot price: $293.32 at 2026-05-10 15:35:22.421186



In [6]:
def fetch_options_for_ticker(ticker: str, verbose: bool = True) -> Optional[pd.DataFrame]:
    """
    Fetch options data for a ticker meeting all strategy criteria.
    
    Criteria:
    ---------
    1. Earnings within DAYS_LOOKAHEAD days
    2. Earnings on a Friday
    3. Earnings BMO or AMC (not during market hours)
    4. Options expiring the same week as earnings
    5. Delta between DELTA_MIN and DELTA_MAX
    
    Returns:
    --------
    DataFrame with options data or None if criteria not met
    """
    try:
        t = yf.Ticker(ticker)
        
        # ── 1. Check earnings date ───────────────────────────────────────────
        earnings_date, timing = get_earnings_info(t)
        
        if earnings_date is None:
            if verbose:
                print(f'  [{ticker}] No earnings date found')
            return None
        
        days_to_earnings = (earnings_date - TODAY).days
        
        if days_to_earnings < 0:
            if verbose:
                print(f'  [{ticker}] Earnings already passed ({earnings_date})')
            return None
        
        if days_to_earnings > DAYS_LOOKAHEAD:
            if verbose:
                print(f'  [{ticker}] Earnings too far ({earnings_date}, {days_to_earnings}d away)')
            return None
        
        # ── 2. Check if earnings on Friday ───────────────────────────────────
        if not is_friday(earnings_date):
            if verbose:
                day_name = earnings_date.strftime('%A')
                print(f'  [{ticker}] Earnings not on Friday ({earnings_date} is {day_name})')
            return None
        
        # ── 3. Check timing (BMO or AMC only) ────────────────────────────────
        if timing == 'During Market':
            if verbose:
                print(f'  [{ticker}] Earnings during market hours')
            return None
        
        # ── 4. Get spot price ────────────────────────────────────────────────
        spot, spot_ts = get_spot_price(t)
        
        if spot is None or spot <= 0:
            if verbose:
                print(f'  [{ticker}] Could not get spot price')
            return None
        
        # ── 5. Find expiration dates in the earnings week ────────────────────
        earnings_week_start = get_week_start(earnings_date)
        
        try:
            all_expirations = t.options
        except:
            if verbose:
                print(f'  [{ticker}] Could not fetch options chain')
            return None
        
        if not all_expirations:
            if verbose:
                print(f'  [{ticker}] No options available')
            return None
        
        # Filter to expirations in the same week as earnings
        target_expirations = []
        for exp_str in all_expirations:
            try:
                exp_date = date.fromisoformat(exp_str)
                if get_week_start(exp_date) == earnings_week_start:
                    target_expirations.append(exp_str)
            except:
                continue
        
        if not target_expirations:
            if verbose:
                print(f'  [{ticker}] No expirations in earnings week ({earnings_date})')
            return None
        
        # ── 6. Fetch options chains and compute Greeks ───────────────────────
        rows = []
        
        for exp_str in target_expirations:
            exp_date = date.fromisoformat(exp_str)
            days_to_expiry = (exp_date - TODAY).days
            T = max(days_to_expiry, 1) / 365.0  # Time to expiry in years
            
            try:
                chain = t.option_chain(exp_str)
            except:
                continue
            
            # Process both calls and puts
            for opt_type, df in [('call', chain.calls), ('put', chain.puts)]:
                if df.empty:
                    continue
                
                for _, row in df.iterrows():
                    # Get implied volatility
                    iv = row.get('impliedVolatility', np.nan)
                    if pd.isna(iv) or iv <= 0:
                        continue
                    
                    strike = row.get('strike', np.nan)
                    if pd.isna(strike) or strike <= 0:
                        continue
                    
                    # Calculate Greeks
                    greeks = bs_greeks(
                        S=spot, K=strike, T=T,
                        r=RISK_FREE_RATE, sigma=iv, 
                        opt_type=opt_type
                    )
                    
                    # Filter by delta (using absolute value for comparison)
                    delta_abs = abs(greeks['delta'])
                    if not (DELTA_MIN <= delta_abs <= DELTA_MAX):
                        continue
                    
                    # Get bid/ask
                    bid = row.get('bid', np.nan)
                    ask = row.get('ask', np.nan)
                    
                    # Calculate mid price
                    if not pd.isna(bid) and not pd.isna(ask) and bid > 0 and ask > 0:
                        mid = (bid + ask) / 2
                    else:
                        mid = row.get('lastPrice', np.nan)
                    
                    rows.append({
                        'ticker': ticker,
                        'spot_price': round(spot, 2),
                        'spot_timestamp': spot_ts,
                        'earnings_date': earnings_date,
                        'earnings_timing': timing,
                        'days_to_earnings': days_to_earnings,
                        'expiration': exp_date,
                        'days_to_expiry': days_to_expiry,
                        'option_type': opt_type,
                        'strike': strike,
                        'bid': bid,
                        'ask': ask,
                        'mid': round(mid, 2) if not pd.isna(mid) else np.nan,
                        'last_price': row.get('lastPrice', np.nan),
                        'implied_vol': round(iv, 4),
                        'delta': greeks['delta'],
                        'gamma': greeks['gamma'],
                        'vega': greeks['vega'],
                        'theta': greeks['theta'],
                        'rho': greeks['rho'],
                        'open_interest': row.get('openInterest', np.nan),
                        'volume': row.get('volume', np.nan),
                        'contract_symbol': row.get('contractSymbol', ''),
                        'in_the_money': row.get('inTheMoney', False),
                    })
        
        if not rows:
            if verbose:
                print(f'  [{ticker}] No options in delta range {DELTA_MIN}-{DELTA_MAX}')
            return None
        
        return pd.DataFrame(rows)
    
    except Exception as e:
        if verbose:
            print(f'  [{ticker}] ERROR: {e}')
        return None


print('fetch_options_for_ticker defined successfully.')

fetch_options_for_ticker defined successfully.


## Scan Universe

In [7]:
# ── Scan the full universe ────────────────────────────────────────────────────

print(f'Scanning {len(TICKERS)} tickers for Friday earnings with options...\n')

results = []
matched_tickers = []
skipped_tickers = []

for i, ticker in enumerate(TICKERS, 1):
    print(f'[{i:2d}/{len(TICKERS)}] {ticker}...', end=' ')
    
    df = fetch_options_for_ticker(ticker, verbose=False)
    
    if df is not None and len(df) > 0:
        results.append(df)
        matched_tickers.append(ticker)
        print(f'✓ {len(df)} contracts matched')
    else:
        skipped_tickers.append(ticker)
        # Get reason for skipping
        t = yf.Ticker(ticker)
        earnings_date, timing = get_earnings_info(t)
        if earnings_date is None:
            print('✗ No earnings date')
        elif not is_friday(earnings_date):
            print(f'✗ Earnings on {earnings_date.strftime("%A")}')
        elif timing == 'During Market':
            print('✗ Earnings during market hours')
        else:
            days_away = (earnings_date - TODAY).days
            if days_away < 0:
                print(f'✗ Earnings passed ({earnings_date})')
            elif days_away > DAYS_LOOKAHEAD:
                print(f'✗ Earnings too far ({days_away}d)')
            else:
                print('✗ No options in delta range or earnings week')
    
    time.sleep(0.3)  # Rate limiting

# Combine results
if results:
    all_options = pd.concat(results, ignore_index=True)
    print(f'\n{"="*60}')
    print(f'TOTAL: {len(all_options)} contracts from {len(matched_tickers)} tickers')
    print(f'Matched tickers: {matched_tickers}')
else:
    all_options = pd.DataFrame()
    print(f'\n{"="*60}')
    print('No matches found in the universe.')

print(f'Skipped: {len(skipped_tickers)} tickers')

Scanning 42 tickers for Friday earnings with options...

[ 1/42] AAPL... ✗ No options in delta range or earnings week
[ 2/42] MSFT... ✗ Earnings on Thursday
[ 3/42] GOOGL... ✗ No options in delta range or earnings week
[ 4/42] AMZN... ✗ No options in delta range or earnings week
[ 5/42] META... ✗ Earnings on Thursday
[ 6/42] NVDA... ✗ Earnings on Thursday
[ 7/42] TSLA... ✗ Earnings on Thursday
[ 8/42] AMD... ✗ Earnings on Wednesday
[ 9/42] INTC... ✗ No options in delta range or earnings week
[10/42] QCOM... ✗ Earnings on Thursday
[11/42] MU... ✗ Earnings on Thursday
[12/42] AVGO... ✗ Earnings on Thursday
[13/42] MRVL... ✗ Earnings on Thursday
[14/42] AMAT... ✓ 11 contracts matched
[15/42] CRM... ✗ Earnings on Thursday
[16/42] ADBE... ✓ 5 contracts matched
[17/42] NFLX... ✓ 2 contracts matched
[18/42] ORCL... ✗ Earnings on Thursday
[19/42] NOW... ✗ Earnings on Thursday
[20/42] SNOW... ✗ Earnings on Thursday
[21/42] PLTR... ✗ Earnings on Tuesday
[22/42] JPM... ✗ Earnings on Tuesday
[23/4

## Results

In [8]:
# ── Display results ───────────────────────────────────────────────────────────

if all_options.empty:
    print('No data to display.')
else:
    display_cols = [
        'ticker', 'spot_price', 'spot_timestamp',
        'earnings_date', 'earnings_timing', 'days_to_earnings',
        'expiration', 'days_to_expiry',
        'option_type', 'strike',
        'bid', 'ask', 'mid',
        'implied_vol',
        'delta', 'gamma', 'vega', 'theta',
        'open_interest', 'volume',
    ]
    
    # Sort by ticker, expiration, option type, and strike
    sorted_df = all_options[display_cols].sort_values(
        ['ticker', 'expiration', 'option_type', 'strike']
    )
    
    print(f'Displaying {len(sorted_df)} options matching criteria:\n')
    display(sorted_df)

Displaying 18 options matching criteria:



,ticker,spot_price,spot_timestamp,earnings_date,earnings_timing,days_to_earnings,expiration,days_to_expiry,option_type,strike,bid,ask,mid,implied_vol,delta,gamma,vega,theta,open_interest,volume
11,ADBE,253.04,2026-05-10 15:35:44.478260,2026-06-12,BMO,33,2026-06-12,33,call,280.0,5.50,6.90,6.20,0.5015,0.2847,0.008893,0.2582,-0.2043,29,8.0
12,ADBE,253.04,2026-05-10 15:35:44.478260,2026-06-12,BMO,33,2026-06-12,33,call,285.0,4.20,5.70,4.95,0.5246,0.2580,0.008095,0.2458,-0.2027,60,4.0
13,ADBE,253.04,2026-05-10 15:35:44.478260,2026-06-12,BMO,33,2026-06-12,33,call,290.0,3.45,4.75,4.10,0.5247,0.2238,0.007490,0.2275,-0.1872,22,33.0
14,ADBE,253.04,2026-05-10 15:35:44.478260,2026-06-12,BMO,33,2026-06-12,33,put,230.0,5.70,7.00,6.35,0.5167,-0.2363,0.007840,0.2345,-0.1755,47,3.0
15,ADBE,253.04,2026-05-10 15:35:44.478260,2026-06-12,BMO,33,2026-06-12,33,put,235.0,7.05,8.30,7.68,0.5056,-0.2778,0.008717,0.2552,-0.1859,30,15.0
0,AMAT,435.44,2026-05-10 15:35:41.410493,2026-05-15,BMO,5,2026-05-15,5,call,465.0,8.35,9.70,9.02,0.8912,0.2838,0.007460,0.1727,-1.5531,51,82.0
1,AMAT,435.44,2026-05-10 15:35:41.410493,2026-05-15,BMO,5,2026-05-15,5,call,467.5,7.70,9.00,8.35,0.8894,0.2662,0.007242,0.1673,-1.5014,3,9.0
2,AMAT,435.44,2026-05-10 15:35:41.410493,2026-05-15,BMO,5,2026-05-15,5,call,470.0,7.00,8.00,7.50,0.8759,0.2458,0.007055,0.1605,-1.4181,3089,281.0
3,AMAT,435.44,2026-05-10 15:35:41.410493,2026-05-15,BMO,5,2026-05-15,5,call,472.5,6.40,7.45,6.93,0.8750,0.2295,0.006802,0.1546,-1.3642,8,413.0
4,AMAT,435.44,2026-05-10 15:35:41.410493,2026-05-15,BMO,5,2026-05-15,5,call,475.0,5.05,7.15,6.10,0.8574,0.2089,0.006575,0.1464,-1.2662,28,146.0


## Analysis

In [9]:
# ── Summary statistics by ticker ──────────────────────────────────────────────

if not all_options.empty:
    summary = (
        all_options
        .groupby(['ticker', 'earnings_date', 'earnings_timing', 'option_type'])
        .agg(
            spot_price=('spot_price', 'first'),
            contracts=('strike', 'count'),
            avg_delta=('delta', lambda x: round(x.abs().mean(), 3)),
            avg_iv=('implied_vol', lambda x: round(x.mean(), 3)),
            avg_mid=('mid', lambda x: round(x.mean(), 2)),
            avg_theta=('theta', lambda x: round(x.mean(), 4)),
            total_oi=('open_interest', 'sum'),
            total_volume=('volume', 'sum'),
        )
        .reset_index()
    )
    
    print('Summary by ticker / option type:')
    print()
    display(summary)
else:
    print('No data available for summary.')

Summary by ticker / option type:



,ticker,earnings_date,earnings_timing,option_type,spot_price,contracts,avg_delta,avg_iv,avg_mid,avg_theta,total_oi,total_volume
0,ADBE,2026-06-12,BMO,call,253.04,3,0.256,0.517,5.08,-0.1981,111,45.0
1,ADBE,2026-06-12,BMO,put,253.04,2,0.257,0.511,7.02,-0.1807,77,18.0
2,AMAT,2026-05-15,BMO,call,435.44,5,0.247,0.878,7.58,-1.4206,3179,931.0
3,AMAT,2026-05-15,BMO,put,435.44,6,0.249,0.865,8.35,-1.3751,2016,391.0
4,NFLX,2026-07-17,BMO,call,87.49,1,0.242,0.372,1.70,-0.0346,47952,6947.0
5,NFLX,2026-07-17,BMO,put,87.49,1,0.234,0.349,2.13,-0.0269,22050,200.0


In [10]:
# ── Potential Credit Spread Analysis ──────────────────────────────────────────
# For earnings plays, we might want to sell options with delta ~0.25

if not all_options.empty:
    print('=== Potential Short Leg Candidates (Delta 0.20-0.30) ===')
    print()
    
    # Best put candidates (for put credit spreads / bull put spreads)
    puts = all_options[all_options['option_type'] == 'put'].copy()
    if not puts.empty:
        puts['delta_abs'] = puts['delta'].abs()
        puts_sorted = puts.sort_values(['ticker', 'delta_abs'], ascending=[True, False])
        
        print('PUT Credit Spread Short Legs (Bullish Strategy):')
        put_display = puts_sorted[[
            'ticker', 'spot_price', 'earnings_date', 'earnings_timing',
            'expiration', 'strike', 'bid', 'ask', 'mid',
            'delta', 'theta', 'implied_vol', 'open_interest'
        ]].head(20)
        display(put_display)
        print()
    
    # Best call candidates (for call credit spreads / bear call spreads)
    calls = all_options[all_options['option_type'] == 'call'].copy()
    if not calls.empty:
        calls['delta_abs'] = calls['delta'].abs()
        calls_sorted = calls.sort_values(['ticker', 'delta_abs'], ascending=[True, False])
        
        print('CALL Credit Spread Short Legs (Bearish Strategy):')
        call_display = calls_sorted[[
            'ticker', 'spot_price', 'earnings_date', 'earnings_timing',
            'expiration', 'strike', 'bid', 'ask', 'mid',
            'delta', 'theta', 'implied_vol', 'open_interest'
        ]].head(20)
        display(call_display)
else:
    print('No data available for spread analysis.')

=== Potential Short Leg Candidates (Delta 0.20-0.30) ===

PUT Credit Spread Short Legs (Bullish Strategy):


,ticker,spot_price,earnings_date,earnings_timing,expiration,strike,bid,ask,mid,delta,theta,implied_vol,open_interest
15,ADBE,253.04,2026-06-12,BMO,2026-06-12,235.0,7.05,8.30,7.68,-0.2778,-0.1859,0.5056,30
14,ADBE,253.04,2026-06-12,BMO,2026-06-12,230.0,5.70,7.00,6.35,-0.2363,-0.1755,0.5167,47
10,AMAT,435.44,2026-05-15,BMO,2026-05-15,415.0,9.50,11.50,10.50,-0.2990,-1.5294,0.8740,52
9,AMAT,435.44,2026-05-15,BMO,2026-05-15,412.5,8.25,10.75,9.50,-0.2775,-1.4648,0.8668,108
8,AMAT,435.44,2026-05-15,BMO,2026-05-15,410.0,7.80,9.75,8.78,-0.2587,-1.4219,0.8713,1399
7,AMAT,435.44,2026-05-15,BMO,2026-05-15,407.5,7.15,8.30,7.73,-0.2362,-1.3298,0.8553,153
6,AMAT,435.44,2026-05-15,BMO,2026-05-15,405.0,6.35,8.00,7.17,-0.2197,-1.2908,0.8643,248
5,AMAT,435.44,2026-05-15,BMO,2026-05-15,402.5,5.75,7.05,6.40,-0.2004,-1.2138,0.8575,56
17,NFLX,87.49,2026-07-17,BMO,2026-07-17,80.0,2.09,2.17,2.13,-0.2341,-0.0269,0.3488,22050



CALL Credit Spread Short Legs (Bearish Strategy):


,ticker,spot_price,earnings_date,earnings_timing,expiration,strike,bid,ask,mid,delta,theta,implied_vol,open_interest
11,ADBE,253.04,2026-06-12,BMO,2026-06-12,280.0,5.50,6.90,6.20,0.2847,-0.2043,0.5015,29
12,ADBE,253.04,2026-06-12,BMO,2026-06-12,285.0,4.20,5.70,4.95,0.2580,-0.2027,0.5246,60
13,ADBE,253.04,2026-06-12,BMO,2026-06-12,290.0,3.45,4.75,4.10,0.2238,-0.1872,0.5247,22
0,AMAT,435.44,2026-05-15,BMO,2026-05-15,465.0,8.35,9.70,9.02,0.2838,-1.5531,0.8912,51
1,AMAT,435.44,2026-05-15,BMO,2026-05-15,467.5,7.70,9.00,8.35,0.2662,-1.5014,0.8894,3
2,AMAT,435.44,2026-05-15,BMO,2026-05-15,470.0,7.00,8.00,7.50,0.2458,-1.4181,0.8759,3089
3,AMAT,435.44,2026-05-15,BMO,2026-05-15,472.5,6.40,7.45,6.93,0.2295,-1.3642,0.8750,8
4,AMAT,435.44,2026-05-15,BMO,2026-05-15,475.0,5.05,7.15,6.10,0.2089,-1.2662,0.8574,28
16,NFLX,87.49,2026-07-17,BMO,2026-07-17,100.0,1.67,1.73,1.70,0.2419,-0.0346,0.3718,47952


In [11]:
# ── Implied Volatility Analysis ───────────────────────────────────────────────

if not all_options.empty:
    print('=== Implied Volatility Analysis ===')
    print()
    
    iv_summary = (
        all_options
        .groupby(['ticker', 'option_type'])
        .agg(
            min_iv=('implied_vol', 'min'),
            avg_iv=('implied_vol', 'mean'),
            max_iv=('implied_vol', 'max'),
            contracts=('strike', 'count'),
        )
        .round(4)
        .reset_index()
    )
    
    print('IV Range by Ticker and Option Type:')
    display(iv_summary)
    
    print()
    print('Highest Average IV (best premium sellers):')
    top_iv = iv_summary.sort_values('avg_iv', ascending=False).head(10)
    display(top_iv)
else:
    print('No data available for IV analysis.')

=== Implied Volatility Analysis ===

IV Range by Ticker and Option Type:


,ticker,option_type,min_iv,avg_iv,max_iv,contracts
0,ADBE,call,0.5015,0.5169,0.5247,3
1,ADBE,put,0.5056,0.5112,0.5167,2
2,AMAT,call,0.8574,0.8778,0.8912,5
3,AMAT,put,0.8553,0.8649,0.8740,6
4,NFLX,call,0.3718,0.3718,0.3718,1
5,NFLX,put,0.3488,0.3488,0.3488,1



Highest Average IV (best premium sellers):


,ticker,option_type,min_iv,avg_iv,max_iv,contracts
2,AMAT,call,0.8574,0.8778,0.8912,5
3,AMAT,put,0.8553,0.8649,0.8740,6
0,ADBE,call,0.5015,0.5169,0.5247,3
1,ADBE,put,0.5056,0.5112,0.5167,2
4,NFLX,call,0.3718,0.3718,0.3718,1
5,NFLX,put,0.3488,0.3488,0.3488,1


In [12]:
# ── Greeks Summary ────────────────────────────────────────────────────────────

if not all_options.empty:
    print('=== Greeks Summary ===')
    print()
    
    greeks_summary = (
        all_options
        .groupby(['ticker', 'option_type'])
        .agg(
            avg_delta=('delta', lambda x: round(x.abs().mean(), 4)),
            avg_gamma=('gamma', 'mean'),
            avg_vega=('vega', 'mean'),
            avg_theta=('theta', 'mean'),
            total_theta=('theta', 'sum'),
        )
        .round(4)
        .reset_index()
    )
    
    print('Average Greeks by Ticker and Option Type:')
    display(greeks_summary)
    
    print()
    print('Best Theta Decay (for premium sellers - most negative theta on short):')
    # For short positions, we want the most negative theta (fast decay)
    best_theta = greeks_summary.sort_values('avg_theta').head(10)
    display(best_theta)
else:
    print('No data available for Greeks summary.')

=== Greeks Summary ===

Average Greeks by Ticker and Option Type:


,ticker,option_type,avg_delta,avg_gamma,avg_vega,avg_theta,total_theta
0,ADBE,call,0.2555,0.0082,0.2438,-0.1981,-0.5942
1,ADBE,put,0.2570,0.0083,0.2448,-0.1807,-0.3614
2,AMAT,call,0.2468,0.0070,0.1603,-1.4206,-7.1030
3,AMAT,put,0.2486,0.0071,0.1606,-1.3751,-8.2505
4,NFLX,call,0.2419,0.0222,0.1179,-0.0346,-0.0346
5,NFLX,put,0.2341,0.0233,0.1158,-0.0269,-0.0269



Best Theta Decay (for premium sellers - most negative theta on short):


,ticker,option_type,avg_delta,avg_gamma,avg_vega,avg_theta,total_theta
2,AMAT,call,0.2468,0.0070,0.1603,-1.4206,-7.1030
3,AMAT,put,0.2486,0.0071,0.1606,-1.3751,-8.2505
0,ADBE,call,0.2555,0.0082,0.2438,-0.1981,-0.5942
1,ADBE,put,0.2570,0.0083,0.2448,-0.1807,-0.3614
4,NFLX,call,0.2419,0.0222,0.1179,-0.0346,-0.0346
5,NFLX,put,0.2341,0.0233,0.1158,-0.0269,-0.0269


## Export Data

In [13]:
# ── Export to CSV ─────────────────────────────────────────────────────────────

if not all_options.empty:
    # Create filename with date
    output_filename = f'greeks_earnings_strategy_{TODAY.isoformat()}.csv'
    
    # Export
    all_options.to_csv(output_filename, index=False)
    
    print(f'Exported {len(all_options)} rows to: {output_filename}')
    print()
    print('Columns exported:')
    for col in all_options.columns:
        print(f'  - {col}')
else:
    print('No data to export.')

Exported 18 rows to: greeks_earnings_strategy_2026-05-10.csv

Columns exported:
  - ticker
  - spot_price
  - spot_timestamp
  - earnings_date
  - earnings_timing
  - days_to_earnings
  - expiration
  - days_to_expiry
  - option_type
  - strike
  - bid
  - ask
  - mid
  - last_price
  - implied_vol
  - delta
  - gamma
  - vega
  - theta
  - rho
  - open_interest
  - volume
  - contract_symbol
  - in_the_money


## Notes

### Data Sources
- **Options Chain**: yfinance (free, real-time delayed)
- **Greeks**: Calculated via Black-Scholes model using IV from yfinance
- **Earnings Dates**: yfinance calendar data

### Strategy Criteria Applied
1. **US Stock Options**: Scanned against a curated universe of liquid US stocks
2. **Same Week as Earnings**: Options expire in the same calendar week as earnings
3. **Earnings on Friday**: Only tickers with Friday earnings are included
4. **BMO/AMC Only**: Earnings during market hours are excluded
5. **Delta 0.20-0.30**: Targets OTM options suitable for credit spread short legs

### Limitations
- yfinance earnings timing (BMO/AMC) may not always be accurate
- Greeks calculated using simple Black-Scholes (no dividend adjustment)
- Spot prices are delayed (not real-time)

### For More Accurate Data
Consider using:
- **CBOE DataShop**: Official options data with Greeks
- **Tradier API**: Real-time options with Greeks
- **ORATS**: Professional options analytics
- **TD Ameritrade / Schwab API**: Real-time data